# 2.1 Travel agent (remote MCP)

Instead of spawning a local process, connect to a **hosted** MCP server over HTTP. [Kiwi.com](https://mcp.kiwi.com) exposes flight-search tools.

`transport: "streamable_http"` + `url` is the remote equivalent of stdio + command.

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Connect to Kiwi MCP

`get_tools()` returns whatever the remote server advertises (search, booking-related helpers, etc.). Network required.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

`checkpointer` keeps the thread if you add follow-ups. The system prompt forbids clarifying questions so the agent must search with the details you gave.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
    config
    )

In [ ]:
from pprint import pprint

# Full trace: MCP tool calls + results
pprint(response)

In [ ]:
print(response["messages"][-1].content)